# Predicción de Consumo de Alcohol — Rusia 2024

Red neuronal profunda para predecir el consumo de alcohol puro per cápita en regiones de Rusia para 2024.

**Ejecutar celdas en orden (▶). No requiere configuración previa.**

## 1. Instalar dependencias + clonar repositorio

In [ ]:
!pip install -q torch pandas numpy scikit-learn matplotlib seaborn
!git clone -q https://github.com/jfelipeq14/alcohol-prediction-russia.git
%cd alcohol-prediction-russia
print("Listo")

## 2. (Opcional) Montar Google Drive

Si quieres guardar los resultados (modelo, CSVs, gráficas) permanentemente en tu Drive, ejecuta esta celda.
Si la omites, los resultados solo estarán disponibles durante esta sesión de Colab.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# Crear carpeta para los resultados
DRIVE_PATH = "/content/drive/MyDrive/alcohol-prediction-rusia"
!mkdir -p "$DRIVE_PATH"
print(f"Resultados se guardarán en: {DRIVE_PATH}")
print("\n💡 Si no quieres usar Drive, omite esta celda (Cmd/Ctrl + Shift + Enter para saltar)")

## 3. Ejecutar pipeline completo

Esto carga los datos, entrena la red neuronal, evalúa y predice 2024.
Tarda ~1-2 minutos (usará GPU si está disponible).

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "/content/alcohol-prediction-russia")

from src.main import main
main()

## 4. Guardar en Drive (opcional)

Si montaste Drive en el paso 2, ejecuta esta celda para copiar los resultados.

In [ ]:
import shutil, os
try:
    dst = DRIVE_PATH
    for f in os.listdir("output"):
        shutil.copy2(f"output/{f}", f"{dst}/{f}")
    print(f"Resultados copiados a {dst}/")
    print(f"Archivos: {os.listdir(dst)}")
except NameError:
    print("Drive no montado. Salta este paso o ejecuta la celda del paso 2 primero.")

## 5. Resultados

Vista previa de las predicciones y rankings.

In [ ]:
import pandas as pd

preds = pd.read_csv("output/predicciones_2024.csv")
display(preds.head(10))
print(f"\nTotal predicciones: {len(preds)}")

In [ ]:
regions = pd.read_csv("output/ranking_regiones.csv")
print("Top 10 regiones:")
display(regions.head(10))
print("\nBottom 5:")
display(regions.tail(5))

In [ ]:
beverages = pd.read_csv("output/ranking_bebidas.csv")
display(beverages)

## 6. Gráficas

In [ ]:
from IPython.display import Image, display

plots = ["training_history.png", "actual_vs_predicted.png",
         "ranking.png", "model_comparison.png"]

for p in plots:
    display(Image(filename=f"output/{p}"))

## 7. Análisis adicional (residuales, distribuciones)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch

from src.config import Config
from src.data.loader import CSVLoader
from src.data.preprocessor import DataPreprocessor
from src.data.splitter import DataSplitter
from src.data.dataset import AlcoholDataset

sns.set_theme(style="whitegrid")

# Reconstruir datos de test
config = Config()
df = CSVLoader(config.data_dir / config.raw_data_file).load()
preprocessor = DataPreprocessor()
pivoted = preprocessor.pivot_data(df)
X, y = preprocessor.extract_train_data(pivoted)
splitter = DataSplitter()
splits = splitter.split(X, y)
X_test = preprocessor.fit_transform(splits["X_train"])
X_test = preprocessor.transform(splits["X_test"])
y_test = splits["y_test"].values

# Cargar modelo
from src.model.architecture import AlcoholPredictor
model = AlcoholPredictor(input_dim=X_test.shape[1])
model.load_state_dict(torch.load("output/best_model.pth", weights_only=True))
model.eval()

y_pred = model(torch.tensor(X_test, dtype=torch.float32)).detach().numpy().flatten()
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(y_test, bins=20, alpha=0.6, label="Real", color="steelblue", edgecolor="white")
axes[0].hist(y_pred, bins=20, alpha=0.6, label="Predicho", color="coral", edgecolor="white")
axes[0].set_xlabel("Litros de alcohol puro per cápita")
axes[0].set_ylabel("Frecuencia")
axes[0].set_title("Distribución: Real vs Predicho")
axes[0].legend()

axes[1].scatter(y_pred, residuals, alpha=0.6, edgecolors="k", linewidth=0.5)
axes[1].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[1].set_xlabel("Predicho")
axes[1].set_ylabel("Residual")
axes[1].set_title("Gráfico de residuales")

plt.tight_layout()
plt.show()

In [ ]:
# Residuales por tipo de bebida
pivoted_test = pivoted.loc[splits["test_idx"]]
beverage_types_test = pivoted_test["Type of alcoholic beverages"].values

fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(x=beverage_types_test, y=residuals, ax=ax, palette="Set2")
ax.axhline(0, color="red", linestyle="--", alpha=0.4)
ax.set_title("Residuales por tipo de bebida")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
print("=== Fin ===")
print(f"Resultados en: output/")
print("\n📌 Para descargar los archivos:")
print("   - Desde el panel de Archivos de Colab (ícono 📁)")
print("   - O ejecutaste el paso 2 + 4 para guardarlos en Drive")